In [0]:
%run /Workspace/de_shared/config



In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Read from Bronze Delta
df_bronze = (
    spark.read
    .format("delta")
    .option("fs.s3a.access.key", AWS_ACCESS_KEY_ID)
    .option("fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
    .option("fs.s3a.endpoint", "s3.amazonaws.com")
    .load("s3a://medinsight-partd-yash/delta/bronze/partd_raw/")
)

print(f"Bronze row count: {df_bronze.count():,}")

Bronze row count: 25,231,862


In [0]:
df_typed = df_bronze.select(
    col("Prscrbr_NPI").alias("prescriber_npi"),
    trim(col("Prscrbr_Last_Org_Name")).alias("prescriber_last_name"),
    trim(col("Prscrbr_First_Name")).alias("prescriber_first_name"),
    trim(col("Prscrbr_State_Abrvtn")).alias("state"),
    trim(col("Prscrbr_Type")).alias("specialty"),
    trim(col("Brnd_Name")).alias("drug_brand"),
    trim(col("Gnrc_Name")).alias("drug_generic"),
    col("Tot_Clms").cast(IntegerType()).alias("total_claims"),
    col("Tot_Day_Suply").cast(IntegerType()).alias("total_day_supply"),
    col("Tot_Drug_Cst").cast(DoubleType()).alias("total_cost_usd"),
    lit(2021).alias("data_year"),
    col("_source_file"),
    col("_ingested_at"),
    col("_batch_id")
)

print(f"Row count after select: {df_typed.count():,}")
df_typed.printSchema()

Row count after select: 25,231,862
root
 |-- prescriber_npi: string (nullable = true)
 |-- prescriber_last_name: string (nullable = true)
 |-- prescriber_first_name: string (nullable = true)
 |-- state: string (nullable = true)
 |-- specialty: string (nullable = true)
 |-- drug_brand: string (nullable = true)
 |-- drug_generic: string (nullable = true)
 |-- total_claims: integer (nullable = true)
 |-- total_day_supply: integer (nullable = true)
 |-- total_cost_usd: double (nullable = true)
 |-- data_year: integer (nullable = false)
 |-- _source_file: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _batch_id: string (nullable = true)



In [0]:
# Bad rows — NPI is null or blank or not 10 digits
df_rejected = df_typed.filter(
    col("prescriber_npi").isNull() |
    (trim(col("prescriber_npi")) == "") |
    (length(trim(col("prescriber_npi"))) != 10) |
    col("total_claims").isNull() |
    (col("total_claims") < 1)
).withColumn("rejection_reason",
    when(col("prescriber_npi").isNull(), "NPI is null")
    .when(trim(col("prescriber_npi")) == "", "NPI is blank")
    .when(length(trim(col("prescriber_npi"))) != 10, "NPI not 10 digits")
    .when(col("total_claims").isNull(), "total_claims is null")
    .when(col("total_claims") < 1, "total_claims less than 1")
).withColumn("_rejected_at", current_timestamp())

# Good rows — everything that is not rejected
df_clean = df_typed.filter(
    col("prescriber_npi").isNotNull() &
    (trim(col("prescriber_npi")) != "") &
    (length(trim(col("prescriber_npi"))) == 10) &
    col("total_claims").isNotNull() &
    (col("total_claims") >= 1)
)

# Fill nulls in non-key columns
df_clean = df_clean \
    .fillna({"prescriber_first_name": "", "specialty": "UNKNOWN"}) \
    .withColumn("total_cost_usd", when(col("total_cost_usd").isNull(), 0.0).otherwise(col("total_cost_usd"))) \
    .withColumn("total_day_supply", when(col("total_day_supply").isNull(), 0).otherwise(col("total_day_supply")))

print(f"Clean rows:    {df_clean.count():,}")
print(f"Rejected rows: {df_rejected.count():,}")

Clean rows:    25,231,862
Rejected rows: 0


In [0]:
from pyspark.sql.window import Window

# Define window - partition by our merge key, order by highest claims
window_spec = Window.partitionBy(
    "prescriber_npi", "drug_generic", "data_year"
).orderBy(col("total_claims").desc())

# Add row number — row 1 = highest claims per NPI+drug+year
df_deduped = df_clean \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .drop("row_num")

print(f"Before dedup: {df_clean.count():,}")
print(f"After dedup:  {df_deduped.count():,}")
print(f"Duplicates removed: {df_clean.count() - df_deduped.count():,}")

Before dedup: 25,231,862
After dedup:  23,850,038
Duplicates removed: 1,381,824


In [0]:
df_silver = df_deduped \
    .withColumn("_updated_at", current_timestamp())

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("fs.s3a.access.key", AWS_ACCESS_KEY_ID) \
    .option("fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY) \
    .option("fs.s3a.endpoint", "s3.amazonaws.com") \
    .partitionBy("data_year") \
    .save("s3a://medinsight-partd-yash/delta/silver/prescriber_claims/")

print(f"Silver write complete: {df_silver.count():,} rows")

Silver write complete: 23,850,038 rows


In [0]:
df_rejected.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("fs.s3a.access.key", AWS_ACCESS_KEY_ID) \
    .option("fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY) \
    .option("fs.s3a.endpoint", "s3.amazonaws.com") \
    .save("s3a://medinsight-partd-yash/delta/rejected/partd_rejected/")

print(f"Rejected write complete: {df_rejected.count():,} rows")

Rejected write complete: 0 rows


In [0]:
df_verify = (
    spark.read
    .format("delta")
    .option("fs.s3a.access.key", AWS_ACCESS_KEY_ID)
    .option("fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
    .option("fs.s3a.endpoint", "s3.amazonaws.com")
    .load("s3a://medinsight-partd-yash/delta/silver/prescriber_claims/")
)

print(f"Silver row count: {df_verify.count():,}")
df_verify.show(3, truncate=False)

Silver row count: 23,850,038
+--------------+--------------------+---------------------+-----+-----------------+-------------------+-------------------+------------+----------------+--------------+---------+----------------------------+-------------------------+------------------------------------+--------------------------+
|prescriber_npi|prescriber_last_name|prescriber_first_name|state|specialty        |drug_brand         |drug_generic       |total_claims|total_day_supply|total_cost_usd|data_year|_source_file                |_ingested_at             |_batch_id                           |_updated_at               |
+--------------+--------------------+---------------------+-----+-----------------+-------------------+-------------------+------------+----------------+--------------+---------+----------------------------+-------------------------+------------------------------------+--------------------------+
|1003000126    |Enkeshafi           |Ardalan              |MD   |Internal Med

In [0]:
# Data quality assertions
assert df_verify.filter(col("prescriber_npi").isNull()).count() == 0, "NPI nulls found"
assert df_verify.filter(col("total_claims") < 1).count() == 0, "Invalid claims found"
assert df_verify.filter(col("total_cost_usd") < 0).count() == 0, "Negative costs found"
assert df_verify.count() == 23850038, "Row count mismatch"

print("All DQ checks passed ✅")

All DQ checks passed ✅
